# AGEMR：按主效应 P 筛选后的线性交互显著比例

本分析回答：**pooled GWAS 主效应越强的 SNP，其中 age interaction 显著的比例是否提高？**

比较全部面板、`P_main < 1e-5` 和 `P_main < 5e-8`。每组计算交互 `P_int < 0.05`、`P_int < 0.05/M` 的数量/比例，以及相对全部可比较 SNP 的比例倍数。`M` 为原完整面板大小（此项目通常 1902）；分组后不缩小 Bonferroni 分母。

`P_main` 来自**没有交互项**的 pooled GWAS。不能用交互模型的 ADD P 代替。这里用已完成 linear run 保存的相同 phenotype / nuisance covariates / genotype 文件另跑 pooled GWAS，避免依赖旧 main-effect 结果的未记录样本变化。首次运行三表型×22 染色体；完整命令和输入 hash 相同的成功结果可复用。

两个筛选集合是嵌套的；比例与倍数只作描述，不报告忽略 LD、同样本主效应筛选和 SNP 选择过程的 enrichment 检验 P。若指的是直接筛 `P_int < 1e-5/5e-8`，在 M=1902 时这些 SNP 本来都满足 nominal 和面板 Bonferroni，条件比例是 100%，不能据此说明 enrichment。

In [ ]:
from pathlib import Path
import json, hashlib, os, shutil, subprocess, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display

# 留空则读取原持久目录下最近完成的一次 linear run。
# 也可以填入上一份 notebook 打印的完整 run 目录。
LINEAR_RUN = None
B1_ROOT = Path.home() / "workspace/agemr-data/final_cohort_v1/B1_age_varying"
PLINK_EXECUTABLE = None  # 默认复用 ~/plink2_env/bin/plink2；不在 workspace 安装软件。
THREADS = 2
MEMORY_MB = 1024
ALPHA = 0.05
THRESHOLDS = [("All eligible", None), ("P_main < 1e-5", 1e-5), ("P_main < 5e-8", 5e-8)]

if LINEAR_RUN is None:
    found = list((B1_ROOT / "B1_linear_age").glob("*/RUN_COMPLETE.json"))
    if not found:
        raise RuntimeError("No completed linear run found; set LINEAR_RUN to its full directory.")
    completion_file = max(found, key=lambda p: json.loads(p.read_text()).get("finished_utc", ""))
    LINEAR_RUN = completion_file.parent
else:
    LINEAR_RUN = Path(LINEAR_RUN).expanduser()
completion = json.loads((LINEAR_RUN / "RUN_COMPLETE.json").read_text())
if completion.get("status") != "complete":
    raise RuntimeError("The selected linear run has not completed.")
M = int(completion["panel_variants"])
TRAITS = completion["traits"]
CHROMS = completion["chromosomes"]
BONF = ALPHA / M
DEST = LINEAR_RUN / "main_effect_stratified_proportions"
DEST.mkdir(exist_ok=True)
print("Using completed linear run:", LINEAR_RUN)
print("Fixed interaction Bonferroni threshold:", BONF)
KEY = ["#CHROM", "POS", "ID", "REF", "ALT"]

def sha(path):
    h=hashlib.sha256()
    with Path(path).open("rb") as f:
        for block in iter(lambda:f.read(1024*1024), b""):
            h.update(block)
    return h.hexdigest()

def read_result(path):
    x = pd.read_csv(path, sep="\t", dtype=str, keep_default_na=False)
    x["#CHROM"] = x["#CHROM"].str.replace(r"^chr", "", regex=True)
    for col in ("POS", "OBS_CT", "P"):
        x[col] = pd.to_numeric(x[col], errors="coerce")
    return x

choices = [PLINK_EXECUTABLE, str(Path.home()/"plink2_env/bin/plink2"), shutil.which("plink2")]
usable = []
for item in choices:
    if not item or not Path(item).is_file():
        continue
    try:
        v = subprocess.run([item,"--version"],capture_output=True,text=True,timeout=30)
    except (OSError,subprocess.TimeoutExpired):
        continue
    if v.returncode == 0 and "PLINK" in (v.stdout+v.stderr):
        usable.append((item,(v.stdout+v.stderr).strip()))
        break
if not usable:
    raise RuntimeError("No executable PLINK2 found. Set PLINK_EXECUTABLE to the working path from the linear notebook.")
PLINK, VERSION = usable[0]
print(VERSION)

## 运行相同调整下的 pooled main-effect GWAS

复用 linear run 的输入，不增加新的 phenotype 筛选或 genotype QC 阈值。对保存的基因型路径检查大小/修改时间，小输入同时核对 SHA256。主效应和交互结果按 chromosome、position、ID、REF、ALT 一对一对齐，检查 OBS_CT，保留失败记录。A1 可改变方向，但此处只按双侧 P 分组，不比较 beta 方向。

In [ ]:
records = json.loads((LINEAR_RUN / "qc/input_files.json").read_text())
pgen_paths = {int(Path(r["path"]).stem.replace("BMI_chr", "")): Path(r["path"])
              for r in records if Path(r["path"]).suffix == ".pgen"}
for record in records:
    path = Path(record["path"])
    if path.suffix not in (".pgen", ".pvar", ".psam"):
        continue
    st = path.stat()
    if st.st_size != record["bytes"] or st.st_mtime_ns != record["mtime_ns"]:
        raise RuntimeError(f"Genotype input changed since linear run: {path}")
    if record.get("sha256") and sha(path) != record["sha256"]:
        raise RuntimeError(f"Genotype metadata changed: {path}")

def run_main(trait):
    pheno = LINEAR_RUN / "inputs" / f"{trait}.pheno.tsv"
    covar = LINEAR_RUN / "inputs" / f"{trait}.covar.tsv"
    meta = json.loads((LINEAR_RUN / "inputs" / f"{trait}.meta.json").read_text())
    if sha(pheno)!=meta["pheno_sha256"] or sha(covar)!=meta["covar_sha256"]:
        raise RuntimeError(f"{trait}: saved linear inputs changed")
    chunks=[]
    for chrom in CHROMS:
        pgen = pgen_paths[chrom]
        out = DEST / "pooled_main" / trait / f"{trait}_chr{chrom}"
        out.parent.mkdir(parents=True,exist_ok=True)
        result_file = Path(str(out) + f".{trait}.glm.linear")
        signature_file = Path(str(out) + ".successful_signature.json")
        cmd = [PLINK, "--pfile", str(pgen.with_suffix("")),
               "--pheno", str(pheno), "--pheno-name", trait, "--covar", str(covar),
               "--glm", "hide-covar", "--no-input-missing-phenotype",
               "--threads", str(THREADS), "--memory", str(MEMORY_MB), "--out", str(out)]
        signature = {"command":cmd,"version":VERSION,"pheno_sha256":meta["pheno_sha256"],"covar_sha256":meta["covar_sha256"]}
        reused = (result_file.is_file() and signature_file.is_file() and
                  json.loads(signature_file.read_text())==signature)
        if not reused:
            if signature_file.exists():
                signature_file.unlink()
            print(f"Pooled main effect: {trait} chr{chrom}",flush=True)
            with Path(str(out)+".stdout.txt").open("w") as fout, Path(str(out)+".stderr.txt").open("w") as ferr:
                r=subprocess.run(cmd,stdout=fout,stderr=ferr)
            if r.returncode != 0:
                raise RuntimeError(f"{trait} chr{chrom}: PLINK failed; inspect {out}.stderr.txt")
            # A completed command may still include per-SNP errors; these are retained below.
            z=read_result(result_file)
            if not z["TEST"].eq("ADD").all():
                raise RuntimeError("Unexpected pooled GWAS TEST rows")
            signature_file.write_text(json.dumps(signature,indent=2))
        chunks.append(read_result(result_file))
    main=pd.concat(chunks,ignore_index=True)
    if main.duplicated(KEY).any():
        raise RuntimeError(f"{trait}: duplicate main-effect variant keys")
    main.to_csv(DEST/f"{trait}_pooled_main.tsv",sep="\t",index=False)
    return main

## 计算筛选后比例及相对全部 SNP 的倍数

以同时具有有效主效应 P、有效交互 P、且两个模型 OBS_CT 相同的 SNP 为可比较集合。每行报告选中的总 SNP 数、有效数、nominal/Bonferroni 数量和比例。空集比例为缺失值，不写成 0%。显著性阈值始终固定为完整原面板的 0.05/M。

In [ ]:
def truth(series):
    return series.astype(str).str.lower().eq("true")

def summarize_groups(x, trait, m):
    threshold=ALPHA/m
    eligible=x["eligible"]
    reference=x.loc[eligible]
    base_nominal=reference["P_int"].lt(ALPHA).mean() if len(reference) else np.nan
    base_bonf=reference["P_int"].lt(threshold).mean() if len(reference) else np.nan
    rows=[]
    for name,cut in THRESHOLDS:
        selected = pd.Series(True,index=x.index) if cut is None else x["P_main"].lt(cut)
        group=x.loc[selected & eligible]
        n=len(group)
        nom=int(group["P_int"].lt(ALPHA).sum())
        bonf=int(group["P_int"].lt(threshold).sum())
        rate_nom=nom/n if n else np.nan
        rate_bonf=bonf/n if n else np.nan
        rows.append({"trait":trait,"group":name,"N_selected":int(selected.sum()),
                     "N_eligible":n,"N_excluded_QC":int((selected & ~eligible).sum()),
                     "N_int_P_lt_005":nom,"pct_int_P_lt_005":100*rate_nom,
                     "N_int_Bonf":bonf,"pct_int_Bonf":100*rate_bonf,
                     "fold_nominal_vs_all":rate_nom/base_nominal if base_nominal>0 else np.nan,
                     "fold_Bonf_vs_all":rate_bonf/base_bonf if base_bonf>0 else np.nan,
                     "fixed_Bonf_threshold":threshold})
    return pd.DataFrame(rows)

all_summaries=[]
for trait in TRAITS:
    interaction=read_result(LINEAR_RUN/f"{trait}_linear_age_all.tsv")
    main=run_main(trait)
    if interaction.duplicated(KEY).any() or len(interaction)!=M:
        raise RuntimeError(f"{trait}: invalid interaction panel")
    main=main.rename(columns={col:col+"_main" for col in main.columns if col not in KEY})
    x=interaction.merge(main,on=KEY,how="outer",validate="one_to_one",indicator=True)
    if not x["_merge"].eq("both").all() or len(x)!=M:
        raise RuntimeError(f"{trait}: main and interaction panels differ")
    x=x.rename(columns={"P":"P_int"})
    x["eligible"]=(truth(x["VALID"]) & x["ERRCODE_main"].eq(".") &
                    x["P_int"].between(0,1) & x["P_main"].between(0,1) &
                    x["OBS_CT"].eq(x["OBS_CT_main"]))
    x["selected_Pmain_1e5"]=x["P_main"].lt(1e-5)
    x["selected_Pmain_5e8"]=x["P_main"].lt(5e-8)
    x.to_csv(DEST/f"{trait}_main_vs_interaction_merged.tsv",sep="\t",index=False)
    for label,cut in [("1e5",1e-5),("5e8",5e-8)]:
        x.loc[x["eligible"] & x["P_main"].lt(cut)].sort_values("P_int").to_csv(
            DEST/f"{trait}_selected_Pmain_{label}.tsv",sep="\t",index=False)
    all_summaries.append(summarize_groups(x,trait,M))
summary=pd.concat(all_summaries,ignore_index=True)
summary.to_csv(DEST/"main_effect_stratified_interaction_proportions.tsv",sep="\t",index=False)
display(summary.round(3))
print("Saved to:",DEST)

In [ ]:
fig,axes=plt.subplots(1,2,figsize=(11,4),sharex=True)
colors={"BMI":"#24618d","SBP":"#c4752f","DBP":"#4d8a70"}
for trait in TRAITS:
    z=summary.loc[summary["trait"].eq(trait)]
    for ax,metric in zip(axes,["pct_int_P_lt_005","pct_int_Bonf"]):
        ax.plot(range(len(z)),z[metric],marker="o",label=trait,color=colors.get(trait))
for ax,title in zip(axes,["Interaction P < 0.05","Interaction P < fixed panel Bonferroni"]):
    ax.set_xticks(range(len(THRESHOLDS)),[s[0] for s in THRESHOLDS])
    ax.set(ylabel="Interaction-significant SNPs (%)",title=title)
    ax.grid(axis="y",alpha=0.2)
axes[0].axhline(5,ls="--",color="grey",lw=1,label="5% reference (not a selection-adjusted null)")
axes[1].legend()
fig.tight_layout()
fig.savefig(DEST/"main_effect_stratified_interaction_proportions.png",dpi=200)
fig.savefig(DEST/"main_effect_stratified_interaction_proportions.pdf")
plt.show()
plt.close(fig)

**验证：** 已完成 3 表型 × 22 染色体合成 workspace 的运行，并核对分组计数、固定 Bonferroni 阈值、空子集和零基线情形。未在用户真实 cohort 上执行。

**如何解读：** 例如某子集 nominal 比例高于 All eligible 行，并且 `fold_nominal_vs_all > 1`，即可描述为这个子集中的交互显著比例较高。Bonferroni 比例同理。不要把子集分母缩小后得到的更宽松阈值与旧阈值混在一起。

若子集 SNP 很少，单个 SNP 就会明显改变百分比；同时看分子和分母。空子集或总体 Bonferroni 命中为 0 时，相应比例或倍数显示 NaN。

本 notebook 可在新 kernel 独立打开，默认选择最近完成的 linear run；也可设置 LINEAR_RUN 指向指定 run。软件调用使用此前已经修复的 driver-local PLINK2。只读取和写入现有 workspace 数据，不下载 controlled data，不做未经执行的结果推断。